#  Linearen Regressionsmodell zur Vorhersage des Bitcoin-Schlusskurses

In [1]:
from prophet import Prophet
import pandas as pd
import matplotlib.pyplot as plt

import plotly.graph_objs as go
from prophet.plot import plot_plotly


# 📥 Daten laden
df = pd.read_csv("../dataset/output/btc_clean.csv", parse_dates=["date_btc"])

# 📥 1. Daten vorbereiten
btc_df = df[["date_btc", "close_btc"]].dropna().copy()
btc_df["date_btc"] = pd.to_datetime(btc_df["date_btc"]).dt.tz_localize(None)  # Zeitzone entfernen
btc_df.columns = ["ds", "y"]


# 📈 2. Modell initialisieren & trainieren
model = Prophet(daily_seasonality=True)
model.fit(btc_df)

# 🔮 3. Zukunftsdaten erzeugen
future = model.make_future_dataframe(periods=90)
forecast = model.predict(future)

# 📊 4. Visualisierung
fig = plot_plotly(model, forecast)
fig.update_layout(
    title="BTC-Preisvorhersage (interaktiv)",
    xaxis_title="Datum",
    yaxis_title="BTC-Preis (USD)",
    template="plotly_dark"
)
fig.show()

# 🔍  Vorhersagewerte anzeigen
forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(90)


21:55:25 - cmdstanpy - INFO - Chain [1] start processing
21:55:41 - cmdstanpy - INFO - Chain [1] done processing


,ds,yhat,yhat_lower,yhat_upper
3805,2025-04-29,97443.998835,91069.700106,103827.040693
3806,2025-04-30,97612.364011,91256.339493,104126.978643
3807,2025-05-01,97610.439261,91280.167147,103935.390216
3808,2025-05-02,97712.259921,90638.816348,103462.953664
3809,2025-05-03,97768.042510,91250.896076,104551.744628
...,...,...,...,...
3890,2025-07-23,101916.513178,94826.188722,107727.309617
3891,2025-07-24,101972.850304,94965.597707,108517.311303
3892,2025-07-25,102138.408257,95443.417375,108305.249990
3893,2025-07-26,102263.976743,95356.908683,108775.761620


# Histogramme
### für Vergleich zwichen Preise und technischen Indikatoren für BTC und XAU nebeneinander

In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import plotly.graph_objs as go
import plotly.subplots as sp


# Daten laden
df = pd.read_csv("../dataset/output/final_merged_technical_dataset.csv", parse_dates=["timestamp"])

# Paare definieren
paare = [
    ("close_btc", "close_xau"),
    ("MACD_btc", "MACD_xau"),
    ("RSI_btc", "RSI_xau"),
    ("MA_20_btc", "MA_20_xau"),
    ("volume_btc", "volume_xau")
]

# Subplots erstellen
fig = sp.make_subplots(rows=len(paare), cols=2, subplot_titles=[
    f"BTC – {btc} | XAU – {xau}" for btc, xau in paare
])

for i, (btc_col, xau_col) in enumerate(paare, start=1):
    # BTC
    fig.add_trace(
        go.Histogram(x=df[btc_col], nbinsx=50, name=f"{btc_col}", marker_color="blue"),
        row=i, col=1
    )
    # XAU
    fig.add_trace(
        go.Histogram(x=df[xau_col], nbinsx=50, name=f"{xau_col}", marker_color="gold"),
        row=i, col=2
    )

# Layout
fig.update_layout(
    height=300 * len(paare),
    width=1000,
    title_text="Histogramm-Vergleich: BTC vs XAU (interaktiv)",
    showlegend=False,
    template="plotly_dark"
)

fig.show()

